# Lab 05 · ReAct agents: reason, act, observe

**Prepared by Mustafa Assaf** · Lecture 2: Chain, call tools, act · 15–20 minute lab

## Inspect a dependency before choosing its resilience policy
**So far:** Lab 04 let a model select one tool and our code checked it.

**New:** a bounded ReAct-style loop: choose an action, run an allowed tool, observe, then choose again. Review the checkout exercise and cascading failures from microservices slides 38–41.

Our specific exercise configuration chooses a **circuit breaker** when repeated failures reach 50%; a bounded retry is considered only for transient failures on an idempotent call. The 50% threshold is illustrative, as in the slide's breaker configuration; production settings require workload evidence. The agent recommends a policy; it does not deploy infrastructure.

The metrics are simulated teaching inputs. Inventory's 5-second value is a peak, not a measured p99; the failure rates and the 50% decision threshold are exercise assumptions. A retry also requires a transient failure and a bounded policy; idempotency alone is insufficient.

**At a glance**

![Lab 05 at a glance: Goal, then Next action (JSON), then Guards (known tool · facts read), then Run the tool (read target metrics), then Observation (added to history). Then back to Next action: repeat · at most 5 turns.](https://mkassaf.github.io/agentic-ai-course/assets/diagrams/05.svg)

## Review your microservices lectures

- **Microservices & Their Patterns**, slides 38, 39, 40, 41.

**Recall:** Cascading failure; Circuit breaker vs retry; Idempotency and graceful degradation.

Before running the example:

1. Why can retries make a slow dependency worse?
2. Why must the non-idempotent Payments charge not be blindly retried?

Slide numbers count from the title slide, including appendices. The small scenarios and code are teaching adaptations, not production services. Use the lecture decks supplied in class; they are not uploaded with this course.

## The idea in context

ReAct-style control flow alternates model actions with tool observations. The example exposes those exchanges without deploying a checkout system.

An agent loop selects its next step from observations; it still needs a bound and allowed tools.

| Key term | Meaning |
| --- | --- |
| **Agent** | A system in which a model chooses its next action in a loop, based on what it has observed. |
| **Observation** | The result of an action, fed back into the next prompt. |
| **History** | Every action and observation so far; the agent's view of the task. |
| **Guard** | A rule in code that the agent cannot override, such as a turn limit. |

## Pattern · One-Step Tool Agent

*The same single-agent pattern as Lab 04, from the [Architectural Patterns for Agentic AI](https://karthikv1392.github.io/agentic-patterns/) field guide; see Lab 04 for the full card. Related: Incremental Tool Agent, Registry & Adapter (Lab 11).*

| | |
| --- | --- |
| **What changes** | Lab 04 made one tool call per request. Here each turn is one One-Step Tool Agent call, and repeating it, with every observation in the history, makes a *ReAct* loop. |
| **Flow** | Goal → Choose action → Tool → Observe → repeat, or Finish |
| **New liability** | A loop can run forever, or finish without successful metrics for the target. The turn limit and the finish guard address both, on top of Lab 04's checks of each request. |

## Setup
Run this cell once to load the course helpers, then run the steps below in order. Practice answers are scripted. For a live model, choose `PROVIDER` (`"freeinference"`, `"gemini"`, or `"deepseek"`), save its key in Colab Secrets (`FREEINFERENCE_API_KEY`, `GEMINI_API_KEY`, or `DEEPSEEK_API_KEY`), and change `MODE` to `"live"`. `ask` stops after 12 calls; run this cell again to reset the count.

In [ ]:
from importlib.util import find_spec
if not all(find_spec(name) for name in ['coursekit']):
    %pip install -q "agentic-ai-coursekit @ https://github.com/mkassaf/agentic-ai-course/archive/9c78f2181b0f793c60b2b081c79ae74276026e7d.zip"
from coursekit import make_ask
MODE = "practice"
PROVIDER = "freeinference"  # or "gemini" or "deepseek"
ask = make_ask(mode=MODE, provider=PROVIDER)

## Step 1 · Give the agent read-only observation tools
Unknown names return errors. A successful metrics record contains the exact dependency name, so the finish guard can distinguish a real observation from a failed lookup.

In [ ]:
import json

METRICS = {
    "Inventory": {
        "service": "Inventory", "failure_rate": 0.7, "peak_ms": 5000,
        "idempotent": True, "transient_failure": True,
    },
    "Payments": {
        "service": "Payments", "failure_rate": 0.01, "p99_ms": 1500,
        "idempotent": False, "transient_failure": True,
    },
}
def list_dependencies(unused_input):
    return list(METRICS)

def read_metrics(name):
    return METRICS.get(name, {"error": "Unknown dependency; call list_dependencies for exact names."})

TOOLS = {"list_dependencies": list_dependencies, "read_metrics": read_metrics}
print(read_metrics("stock"))

## Step 2 · Ask for an action using the whole history
The model receives the goal, tools and exercise policy. The scripted practice path lists dependencies, reads Inventory and recommends a breaker. The prompt asks for a short action, not a hidden reasoning trace. `iter` creates the practice sequence, and `next(script, "")` takes its next response. Rerun this cell before repeating the loop: an exhausted iterator does not restart itself. In live mode those example strings are ignored.

In [ ]:
target = "Inventory"
POLICY = (
    "For this exercise: failure_rate >= 0.5 -> circuit breaker; "
    "otherwise idempotent AND transient_failure -> bounded retry; otherwise no automatic retry. "
    "Never infer metrics: read the target's record first."
)
script = iter([
    '{"action":"list_dependencies","input":""}',
    '{"action":"read_metrics","input":"Inventory"}',
    '{"action":"finish","input":"circuit breaker"}',
])

def next_action(history):
    prompt = (f"{POLICY}\nGoal: recommend a resilience policy for {target}.\n"
              "Tools: list_dependencies (input ignored), read_metrics (input: exact service name).\n"
              f"History: {json.dumps(history)}\n"
              'JSON only: {"action":"tool name or finish","input":"service name or policy label"}')
    practice_reply = next(script, "")
    step = json.loads(ask(prompt, example=practice_reply))
    if not isinstance(step, dict):
        raise ValueError("Expected an action object.")
    for key in ("action", "input"):
        if not isinstance(step.get(key), str):
            raise ValueError("Action and input must be text.")
    return step

## Step 3 · Guard the loop
A failed lookup must not count as having read metrics. Finishing is allowed only after a successful metrics observation for the target dependency. Unknown tools become observations, and five turns bound the workflow. `observed` starts false and becomes true only after reading the target's facts. `history` stores actions and observations for the next prompt; `break` ends the loop when a valid finish action arrives.

In [ ]:
history = []
decision = None
observed = False
for turn in range(1, 6):
    step = next_action(history)
    action = step["action"]
    value = step["input"]
    print("Turn", turn, action, value)
    if action == "finish":
        if not observed:
            raise ValueError("Read the target's metrics successfully before finishing.")
        decision = value.strip().lower()
        break
    tool = TOOLS.get(action)
    observation = tool(value) if tool else {"error": "Unknown action; use an allowed tool."}
    if action == "read_metrics" and isinstance(observation, dict):
        if observation.get("service") == target and "failure_rate" in observation:
            observed = True
    history.append({"action": action, "input": value, "observation": observation})
    print("Observed:", observation)
assert decision is not None, "Turn budget exhausted; escalate instead of guessing."

## Step 4 · Check the recommendation and its limits
The independent checker uses the real target record. Choosing a breaker does not remove the need for timeouts, isolation and an appropriate fallback.

In [ ]:
facts = METRICS[target]
if facts["failure_rate"] >= 0.5:
    expected = "circuit breaker"
elif facts["idempotent"] and facts["transient_failure"]:
    expected = "bounded retry"
else:
    expected = "no automatic retry"

print("Recommendation:", decision)
print("Exercise expects:", expected, "| tool calls:", len(history))
assert decision == expected, "Recommendation fails the exercise policy."
print("Blindly retrying Payments can duplicate its non-idempotent charge.")

## What you should see
The unknown-name lookup prints an error. The loop lists dependencies, reads Inventory's real metrics, then chooses `circuit breaker`. A failed lookup or another service's metrics cannot unlock `finish`.

## Try it
1. In Step 2's practice script, replace the Inventory lookup with `stock`. Rerun Steps 2–3. The error observation must not unlock `finish`.
2. In Step 2, change `target` to `"Payments"`, the `read_metrics` input to `"Payments"`, and the finish input to `"no automatic retry"`. Rerun Steps 2–4.
3. In Step 1, change Inventory's failure rate to 0.1. Rerun Step 1, then change Step 2's finish input to `"bounded retry"` and rerun Steps 2–4. Explain why both idempotency and a transient fault matter.
4. Pick a timeout, breaker and fallback for the slide's optional Recommendations dependency. Which user-visible feature degrades?

## Takeaway
An agent selects steps from observations, while code controls available actions, successful reads, budgets and final checks. Resilience patterns address different failure modes; indiscriminate retries can amplify an outage or duplicate a charge.